# External immune-aging clock comparison

Paper: Fig. 3f. Source: `final/fig3.ipynb`, cell 192 (the compact final layout). Input: the original aggregate `benchmark.csv`, with `model` and Sound_Life/WangYF_NI `R2`, `MAE`, `PCC`, `MAD` columns. Environment: Python, NumPy, pandas and Matplotlib; Arial fonts are configured separately. Outputs: compact PNG/PDF and `benchmark_with_rank.csv` below `OUTPUT_DIR`. The original ranking directions, missing-dataset handling and four-metric averages are retained. No prior notebook or model is needed **for rendering**. This renders existing benchmark results; it does not rerun competing clocks or validate their predictions. Competitor models and independent environments (including scAgeClock's own environment and siAge's R steps) are not bundled.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Only the bundled `resources/benchmark/benchmark.csv` and licensed Arial fonts; [processed resources](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices). External cohorts are not read for this plot.

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
BENCHMARK_CSV = Path(os.environ.get("PERICLOCK_BENCHMARK_CSV", str(PROJECT_ROOT / "resources/benchmark/benchmark.csv"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/03_external_benchmark"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [BENCHMARK_CSV]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.patches import Rectangle

# ============================================================
# 0. 路径
# ============================================================
file_path = str(BENCHMARK_CSV)
save_dir = str(OUTPUT_DIR)

png_path = os.path.join(save_dir, "benchmark_external_validation_compact.png")
pdf_path = os.path.join(save_dir, "benchmark_external_validation_compact.pdf")
rank_path = os.path.join(save_dir, "benchmark_with_rank.csv")

os.makedirs(save_dir, exist_ok=True)

# ============================================================
# 1. Arial
# ============================================================
font_dir = str(FONT_DIR)

if os.path.exists(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ============================================================
# 2. 读取数据
# ============================================================
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()
df["model"] = df["model"].astype(str).str.strip()

datasets = ["Sound_Life", "WangYF_NI"]

display_names = {
    "Sound_Life": "Sound Life",
    "WangYF_NI": "WangYF_NI"
}

metrics = ["R2", "MAE", "PCC", "MAD"]

for ds in datasets:
    for metric in metrics:
        df[f"{ds}_{metric}"] = pd.to_numeric(
            df[f"{ds}_{metric}"],
            errors="coerce"
        )

# ============================================================
# 3. 排名
# R2 / PCC 越大越好
# MAE / MAD 越小越好
# ============================================================
for ds in datasets:
    rank_cols = []

    for metric in metrics:
        col = f"{ds}_{metric}"
        rank_col = f"{ds}_{metric}_rank"

        ascending = metric in ["MAE", "MAD"]

        df[rank_col] = df[col].rank(
            ascending=ascending,
            method="average",
            na_option="keep"
        )

        rank_cols.append(rank_col)

    df[f"{ds}_score"] = df[rank_cols].mean(
        axis=1,
        skipna=True
    )

    df[f"{ds}_rank"] = df[f"{ds}_score"].rank(
        ascending=True,
        method="min",
        na_option="keep"
    )

# ============================================================
# 4. Overall rank
# ============================================================
df["Overall_score"] = df[
    ["Sound_Life_score", "WangYF_NI_score"]
].mean(axis=1, skipna=True)

df["Overall_rank"] = df["Overall_score"].rank(
    ascending=True,
    method="min",
    na_option="keep"
)

df = df.sort_values(
    ["Overall_score", "model"],
    ascending=[True, True]
).reset_index(drop=True)

print("\nFinal ranking:")
print(
    df[
        [
            "model",
            "Overall_rank",
            "Sound_Life_rank",
            "WangYF_NI_rank"
        ]
    ].to_string(index=False)
)

# ============================================================
# 5. 配色
# ============================================================
methods_color = "#4F6F82"   # 蓝灰
overall_color = "#C96464"   # 柔和砖红
sound_color = "#6F9FC1"     # 低饱和蓝
wang_color = "#E1A15A"      # 低饱和橙

dataset_colors = {
    "Sound_Life": sound_color,
    "WangYF_NI": wang_color
}

def lighten_color(color, amount=0.72):
    rgb = np.array(mpl.colors.to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)

def fmt(v):
    if pd.isna(v):
        return ""
    return f"{v:.2f}"

# ============================================================
# 6. 模型名称自动换行
# ============================================================
def wrap_model_name(name, width=17):

    # 指定名称手动换行
    if name == "IDEAL-Age(DeepSets)":
        return "IDEAL-Age\n(DeepSets)"

    name2 = name.replace("_", "_ ")

    lines = textwrap.wrap(
        name2,
        width=width,
        break_long_words=False,
        break_on_hyphens=False
    )

    lines = [
        x.replace("_ ", "_")
        for x in lines
    ]

    if len(lines) > 2:
        lines = [
            lines[0],
            "".join(lines[1:])
        ]

    return "\n".join(lines)

df["model_display"] = df["model"].apply(
    wrap_model_name
)

# ============================================================
# 7. 指标范围
# ============================================================
ranges = {}

for ds in datasets:
    for metric in metrics:
        vals = df[f"{ds}_{metric}"].dropna()

        if len(vals) > 0:
            ranges[(ds, metric)] = (
                float(vals.min()),
                float(vals.max())
            )

# ============================================================
# 8. 布局
# ============================================================
n = len(df)

method_w = 1.45
overall_w = 0.46

rank_w = 0.40

r2_w = 0.56
pcc_w = 0.56

mae_w = 0.54
mad_w = 0.54

dataset_w = (
    rank_w +
    r2_w +
    mae_w +
    pcc_w +
    mad_w
)

total_w = (
    method_w +
    overall_w +
    dataset_w * 2
)

row_h = 0.68

header1_h = 0.46
header2_h = 0.40

top_margin = 0.10
bottom_margin = 0.12

total_h = (
    top_margin +
    header1_h +
    header2_h +
    n * row_h +
    bottom_margin
)

fig = plt.figure(
    figsize=(
        6.8,
        max(5.8, 0.43 * n + 1.5)
    )
)

ax = fig.add_axes(
    [0.012, 0.025, 0.976, 0.95]
)

ax.set_xlim(0, total_w)
ax.set_ylim(0, total_h)
ax.axis("off")

# ============================================================
# 9. 一级表头
# ============================================================
y1 = total_h - top_margin - header1_h

# Methods
ax.add_patch(
    Rectangle(
        (0, y1),
        method_w,
        header1_h,
        facecolor=methods_color,
        edgecolor="white",
        linewidth=1.2
    )
)

ax.text(
    method_w / 2,
    y1 + header1_h / 2,
    "Methods",
    ha="center",
    va="center",
    fontsize=10.5,
    color="white",
    fontweight="bold"
)

# Overall
x = method_w

ax.add_patch(
    Rectangle(
        (x, y1),
        overall_w,
        header1_h,
        facecolor=overall_color,
        edgecolor="white",
        linewidth=1.2
    )
)

ax.text(
    x + overall_w / 2,
    y1 + header1_h / 2,
    "Overall",
    ha="center",
    va="center",
    fontsize=8.5,
    color="white",
    fontweight="bold"
)

x += overall_w

# Dataset headers
for ds in datasets:
    color = dataset_colors[ds]

    ax.add_patch(
        Rectangle(
            (x, y1),
            dataset_w,
            header1_h,
            facecolor=color,
            edgecolor="white",
            linewidth=1.2
        )
    )

    ax.text(
        x + dataset_w / 2,
        y1 + header1_h / 2,
        display_names[ds],
        ha="center",
        va="center",
        fontsize=10.5,
        color="white",
        fontweight="bold"
    )

    x += dataset_w

# ============================================================
# 10. 二级表头
# ============================================================
y2 = y1 - header2_h

ax.text(
    method_w + overall_w / 2,
    y2 + header2_h / 2,
    "Rank",
    ha="center",
    va="center",
    fontsize=9.0,
    fontweight="bold"
)

x = method_w + overall_w

for ds in datasets:
    for label, width in [
        ("Rank", rank_w),
        ("R²", r2_w),
        ("MAE", mae_w),
        ("PCC", pcc_w),
        ("MAD", mad_w)
    ]:

        ax.text(
            x + width / 2,
            y2 + header2_h / 2,
            label,
            ha="center",
            va="center",
            fontsize=9.0,
            fontweight="bold"
        )

        x += width

# ============================================================
# 11. 主体
# ============================================================
body_top = y2

for i, row in df.iterrows():

    y = body_top - (i + 1) * row_h

    # --------------------------------------------------------
    # 交替灰背景
    # --------------------------------------------------------
    if i % 2 == 1:
        ax.add_patch(
            Rectangle(
                (0, y),
                total_w,
                row_h,
                facecolor="#E5E5E5",
                edgecolor="none"
            )
        )

    # ========================================================
    # Methods
    # ========================================================
    ax.text(
        0.045,
        y + row_h / 2,
        row["model_display"],
        ha="left",
        va="center",
        fontsize=9.4,
        fontweight="bold",
        linespacing=0.90
    )

    # ========================================================
    # Overall rank
    # ========================================================
    x = method_w

    rank = row["Overall_rank"]

    if not pd.isna(rank):

        max_rank = df[
            "Overall_rank"
        ].dropna().max()

        strength = (
            1 -
            (rank - 1) /
            max(max_rank - 1, 1)
        )

        bw = overall_w * (
            0.17 +
            0.73 * strength
        )

        ax.add_patch(
            Rectangle(
                (
                    x + 0.012,
                    y + row_h * 0.14
                ),
                bw,
                row_h * 0.72,
                facecolor=lighten_color(
                    overall_color,
                    0.15 +
                    0.55 *
                    (1 - strength)
                ),
                edgecolor="#555555",
                linewidth=0.55
            )
        )

        if rank <= 3:
            ax.text(
                x + overall_w / 2,
                y + row_h / 2,
                str(int(rank)),
                ha="center",
                va="center",
                fontsize=9.1,
                fontweight="bold"
            )

    x += overall_w

    # ========================================================
    # 两个 dataset
    # ========================================================
    for ds in datasets:

        color = dataset_colors[ds]

        light = lighten_color(
            color,
            0.72
        )

        # ====================================================
        # Rank
        # ====================================================
        ds_rank = row[
            f"{ds}_rank"
        ]

        if not pd.isna(ds_rank):

            max_rank = df[
                f"{ds}_rank"
            ].dropna().max()

            strength = (
                1 -
                (ds_rank - 1) /
                max(max_rank - 1, 1)
            )

            bw = rank_w * (
                0.15 +
                0.76 * strength
            )

            ax.add_patch(
                Rectangle(
                    (
                        x + 0.010,
                        y + row_h * 0.14
                    ),
                    bw,
                    row_h * 0.72,
                    facecolor=light,
                    edgecolor="#666666",
                    linewidth=0.5
                )
            )

            if ds_rank <= 3:
                ax.text(
                    x + rank_w / 2,
                    y + row_h / 2,
                    str(int(ds_rank)),
                    ha="center",
                    va="center",
                    fontsize=9.0,
                    fontweight="bold"
                )

        x += rank_w

        # ====================================================
        # R²
        # 使用 scatter，因此始终是正圆
        # ====================================================
        value = row[
            f"{ds}_R2"
        ]

        if not pd.isna(value):

            vmin, vmax = ranges[
                (ds, "R2")
            ]

            if vmax > vmin:
                norm = (
                    (value - vmin) /
                    (vmax - vmin)
                )
            else:
                norm = 1

            norm = np.clip(
                norm,
                0,
                1
            )

            bubble_x = x + 0.085
            number_x = x + 0.20

            # scatter 的 s 是面积，marker 不会被横纵比例拉伸
            bubble_size = (
                8 +
                42 * norm
            )

            ax.scatter(
                bubble_x,
                y + row_h / 2,
                s=bubble_size,
                marker="o",
                facecolor=light,
                edgecolor=color,
                linewidth=0.85,
                zorder=5
            )

            ax.text(
                number_x,
                y + row_h / 2,
                fmt(value),
                ha="left",
                va="center",
                fontsize=9.0,
                fontweight="bold"
            )

        x += r2_w

        # ====================================================
        # MAE
        # ====================================================
        value = row[
            f"{ds}_MAE"
        ]

        if not pd.isna(value):

            vmin, vmax = ranges[
                (ds, "MAE")
            ]

            if vmax > vmin:
                norm = (
                    (value - vmin) /
                    (vmax - vmin)
                )
            else:
                norm = 0

            norm = np.clip(
                norm,
                0,
                1
            )

            bar_x = x + 0.025
            max_bar = 0.065

            bw = (
                0.012 +
                max_bar * norm
            )

            ax.add_patch(
                Rectangle(
                    (
                        bar_x,
                        y + row_h * 0.18
                    ),
                    bw,
                    row_h * 0.64,
                    facecolor=light,
                    edgecolor=color,
                    linewidth=0.55
                )
            )

            ax.text(
                x + 0.115,
                y + row_h / 2,
                fmt(value),
                ha="left",
                va="center",
                fontsize=9.0,
                fontweight="bold"
            )

        x += mae_w

        # ====================================================
        # PCC
        # 使用 scatter，因此始终是正圆
        # ====================================================
        value = row[
            f"{ds}_PCC"
        ]

        if not pd.isna(value):

            vmin, vmax = ranges[
                (ds, "PCC")
            ]

            if vmax > vmin:
                norm = (
                    (value - vmin) /
                    (vmax - vmin)
                )
            else:
                norm = 1

            norm = np.clip(
                norm,
                0,
                1
            )

            bubble_x = x + 0.085
            number_x = x + 0.20

            bubble_size = (
                8 +
                42 * norm
            )

            ax.scatter(
                bubble_x,
                y + row_h / 2,
                s=bubble_size,
                marker="o",
                facecolor=light,
                edgecolor=color,
                linewidth=0.85,
                zorder=5
            )

            ax.text(
                number_x,
                y + row_h / 2,
                fmt(value),
                ha="left",
                va="center",
                fontsize=9.0,
                fontweight="bold"
            )

        x += pcc_w

        # ====================================================
        # MAD
        # ====================================================
        value = row[
            f"{ds}_MAD"
        ]

        if not pd.isna(value):

            vmin, vmax = ranges[
                (ds, "MAD")
            ]

            if vmax > vmin:
                norm = (
                    (value - vmin) /
                    (vmax - vmin)
                )
            else:
                norm = 0

            norm = np.clip(
                norm,
                0,
                1
            )

            bar_x = x + 0.025
            max_bar = 0.065

            bw = (
                0.012 +
                max_bar * norm
            )

            ax.add_patch(
                Rectangle(
                    (
                        bar_x,
                        y + row_h * 0.18
                    ),
                    bw,
                    row_h * 0.64,
                    facecolor=light,
                    edgecolor=color,
                    linewidth=0.55
                )
            )

            ax.text(
                x + 0.115,
                y + row_h / 2,
                fmt(value),
                ha="left",
                va="center",
                fontsize=9.0,
                fontweight="bold"
            )

        x += mad_w

# ============================================================
# 12. 数据集分隔线
# ============================================================
separator_x = [
    method_w,
    method_w + overall_w,
    method_w + overall_w + dataset_w,
    total_w
]

for xx in separator_x:

    ax.plot(
        [xx, xx],
        [
            body_top - n * row_h,
            y1 + header1_h
        ],
        color="white",
        linewidth=1.1,
        zorder=20
    )

# 底边
ax.plot(
    [0, total_w],
    [
        body_top - n * row_h,
        body_top - n * row_h
    ],
    color="#555555",
    linewidth=0.65
)

# ============================================================
# 13. 保存
# ============================================================
plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor="white"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor="white"
)

plt.show()

# ============================================================
# 14. 保存排名表
# ============================================================
out_cols = [
    "model",
    "Overall_rank",

    "Sound_Life_rank",
    "Sound_Life_R2",
    "Sound_Life_MAE",
    "Sound_Life_PCC",
    "Sound_Life_MAD",

    "WangYF_NI_rank",
    "WangYF_NI_R2",
    "WangYF_NI_MAE",
    "WangYF_NI_PCC",
    "WangYF_NI_MAD"
]

df[out_cols].to_csv(
    rank_path,
    index=False
)

print("\nSaved:")
print(png_path)
print(pdf_path)
print(rank_path)